# Encoding — UCI Drug Consumption Dataset

This notebook encodes the ordinal drug usage labels (CL0–CL6) into a binary target variable for cannabis risk prediction, and saves train/test splits ready for modelling.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('uci_drug_clean.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# The drug columns use ordinal class labels from the UCI study:
#   CL0 = Never used
#   CL1 = Over a decade ago
#   CL2 = Last decade
#   CL3 = Last year
#   CL4 = Last month
#   CL5 = Last week
#   CL6 = Last day

# We focus on Cannabis as the target drug.
# Rationale: cannabis is the most commonly used illicit drug among students,
# has the largest and most balanced class distribution in this dataset,
# and is directly relevant to student drug addiction risk prediction.

print('Cannabis class distribution:')
print(df['Cannabis'].value_counts().sort_index())

In [ ]:
# Binarise the Cannabis target:
#   Non-user (0) = CL0 (never used) + CL1 (used over a decade ago)
#   At Risk  (1) = CL2 through CL6 (used within the last decade or more recently)
#
# Justification: CL2 onwards indicates ongoing or recent drug involvement,
# which aligns with the project objective of identifying at-risk students.

df['cannabis_risk'] = df['Cannabis'].apply(lambda x: 0 if x in ['CL0', 'CL1'] else 1)

print('Binary cannabis_risk distribution:')
print(df['cannabis_risk'].value_counts())
print()
print(f"Non-user (0): {(df['cannabis_risk']==0).sum()} ({(df['cannabis_risk']==0).mean()*100:.1f}%)")
print(f"At Risk  (1): {(df['cannabis_risk']==1).sum()} ({(df['cannabis_risk']==1).mean()*100:.1f}%)")

- The binary encoding produces a moderately imbalanced dataset (~33% Non-user, ~67% At Risk).
- This imbalance will be handled during modelling using `class_weight='balanced'` and sample weighting.

In [ ]:
# Define features and target
FEATURES = ['Age', 'Gender', 'Education', 'Country', 'Ethnicity',
            'Nscore', 'Escore', 'Oscore', 'Ascore', 'Cscore', 'Impulsive', 'SS']

X = df[FEATURES]
y = df['cannabis_risk']

print('Features:', FEATURES)
print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# Drop the raw drug columns — not used as features, only cannabis_risk is the target
# Keep only features + target for the final encoded dataset

df_encoded = df[FEATURES + ['cannabis_risk']].copy()
print('Encoded dataset shape:', df_encoded.shape)
df_encoded.head()

In [ ]:
# Stratified 80/20 train/test split
# stratify=y ensures class proportions are preserved in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {len(X_train)} samples')
print(f'Test : {len(X_test)} samples')
print()
print('Train class distribution:')
print(y_train.value_counts())
print()
print('Test class distribution:')
print(y_test.value_counts())

In [ ]:
# Save train and test splits
train_df = X_train.copy()
train_df['cannabis_risk'] = y_train.values

test_df = X_test.copy()
test_df['cannabis_risk'] = y_test.values

train_df.to_csv('uci_cannabis_train.csv', index=False)
test_df.to_csv('uci_cannabis_test.csv', index=False)

print('Saved: uci_cannabis_train.csv')
print('Saved: uci_cannabis_test.csv')